# Sentiment Analysis with RNN and LSTM
### Following the data through every layer

We'll build a tiny sentiment classifier: a review goes in, and the model says positive (1) or negative (0).

The code stays simple: prepare the data, pad it, stack layers in a `Sequential` model, and call `model.summary()`. The markdown cells explain **what each layer receives, what it does inside, and what it hands to the next layer**, so the shapes in `model.summary()` make sense.

## Four words you need

| Word | Meaning | In this notebook |
|---|---|---|
| **Batch** | how many reviews go into the model together | 12 (all reviews), or 4 during training |
| **Time steps** | how many words each review has after padding | 5 |
| **Features** | how many numbers describe one word | 8 (the embedding size) |
| **Units** | how many numbers the RNN/LSTM uses as its memory | 4 |

Every RNN and LSTM layer in Keras takes input in this shape:

```
(batch, timesteps, features)
```

In `model.summary()` you'll see the batch written as **`None`**. That just means "any batch size": the same model works for 1 review or 1,000.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Flatten, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

tf.keras.utils.set_random_seed(42)   # so everyone in class gets the same results

---
# Part 1: The data

12 short movie reviews: 6 positive (1) and 6 negative (0). Tiny on purpose, so we can see everything.

In [3]:
reviews = [
    "the movie was great",
    "the story was very good",
    "i loved the acting",
    "a great and fun movie",
    "the movie was not good",
    "the story was boring",
    "i hated the acting",
    "a boring and slow movie",
    "very good fun movie",
    "not great at all",
    "the acting was very good",
    "slow story and bad acting",
]

labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0])

print(len(reviews), "reviews")

12 reviews


## Step 1: Words to numbers

A neural network can only work with numbers, so every word gets an ID. I sort the words alphabetically and number them from **1**, because **0 is kept for padding**.

We get 20 unique words, so IDs go from 1 to 20. Including the padding ID 0, the vocabulary size is **21**. We'll give that number to the Embedding layer.

| Word | a | acting | all | and | at | bad | boring | fun | good | great | hated | i | loved | movie | not | slow | story | the | very | was |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **ID** | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 | 20 |

So `"the movie was not good"` becomes `[18, 14, 20, 15, 9]`.

In [4]:
all_words = sorted(set(" ".join(reviews).split()))
word_to_id = {word: i + 1 for i, word in enumerate(all_words)}   # IDs start at 1, 0 is for padding

vocab_size = len(word_to_id) + 1
print("Vocabulary size:", vocab_size)
print(word_to_id)

Vocabulary size: 21
{'a': 1, 'acting': 2, 'all': 3, 'and': 4, 'at': 5, 'bad': 6, 'boring': 7, 'fun': 8, 'good': 9, 'great': 10, 'hated': 11, 'i': 12, 'loved': 13, 'movie': 14, 'not': 15, 'slow': 16, 'story': 17, 'the': 18, 'very': 19, 'was': 20}


In [5]:
sequences = [[word_to_id[word] for word in review.split()] for review in reviews]

for review, seq in zip(reviews, sequences):
    print(f"{review:28} -> {seq}")

the movie was great          -> [18, 14, 20, 10]
the story was very good      -> [18, 17, 20, 19, 9]
i loved the acting           -> [12, 13, 18, 2]
a great and fun movie        -> [1, 10, 4, 8, 14]
the movie was not good       -> [18, 14, 20, 15, 9]
the story was boring         -> [18, 17, 20, 7]
i hated the acting           -> [12, 11, 18, 2]
a boring and slow movie      -> [1, 7, 4, 16, 14]
very good fun movie          -> [19, 9, 8, 14]
not great at all             -> [15, 10, 5, 3]
the acting was very good     -> [18, 2, 20, 19, 9]
slow story and bad acting    -> [16, 17, 4, 6, 2]


## Step 2: Padding, so every review has the same length

Reviews have 4 or 5 words, but a model needs a clean rectangle of numbers. `pad_sequences` makes every review **5 words** long by adding 0s at the **front** (the default `padding='pre'`).

| Review | IDs | Padded |
|---|---|---|
| the movie was great | [18, 14, 20, 10] | [0, 18, 14, 20, 10] |
| the story was very good | [18, 17, 20, 19, 9] | [18, 17, 20, 19, 9] |
| i loved the acting | [12, 13, 18, 2] | [0, 12, 13, 18, 2] |
| a great and fun movie | [1, 10, 4, 8, 14] | [1, 10, 4, 8, 14] |
| the movie was not good | [18, 14, 20, 15, 9] | [18, 14, 20, 15, 9] |

**Why the front?** An RNN reads left to right, and we use its memory at the **end** to decide the sentiment. With padding at the front, the real words sit right before that final decision.

After padding, our data is a matrix of shape **(12, 5)**: 12 reviews (batch) × 5 words (time steps). Read it like a timetable: rows are reviews, columns are time steps, and the RNN walks across the columns.

In [6]:
X = pad_sequences(sequences, maxlen=5)

print(X)
print("Shape:", X.shape, "-> (batch, timesteps)")

[[ 0 18 14 20 10]
 [18 17 20 19  9]
 [ 0 12 13 18  2]
 [ 1 10  4  8 14]
 [18 14 20 15  9]
 [ 0 18 17 20  7]
 [ 0 12 11 18  2]
 [ 1  7  4 16 14]
 [ 0 19  9  8 14]
 [ 0 15 10  5  3]
 [18  2 20 19  9]
 [16 17  4  6  2]]
Shape: (12, 5) -> (batch, timesteps)


---
# Part 2: What each layer does

## The Embedding layer

**Receives:** `(batch, 5)`, word IDs.

A single ID like `9` tells the network nothing about the word "good". The Embedding layer is a **lookup table** with one row per word ID. Each row is a vector of 8 numbers, and the model **learns** these numbers during training, so words used in similar ways end up with similar vectors.

**Does:** replaces every ID with its row from the table.

**Hands back:** `(batch, 5, 8)`. Each review is now 5 vectors of 8 numbers, which is exactly the `(batch, timesteps, features)` shape an RNN needs.

**Parameters:** the table itself = vocab size × embedding size = 21 × 8 = **168**.

```
one review  :  [18, 14, 20, 15, 9]            shape (5,)
after lookup:  5 rows of 8 numbers each         shape (5, 8)
whole batch :  (12, 5)  ->  (12, 5, 8)
```

## The SimpleRNN layer

**Receives:** `(batch, 5, 8)`: for every review, 5 word vectors of 8 numbers.

### What happens inside

The RNN reads each review **one word at a time**, left to right. It carries a **memory vector** $h$ whose size is the number of **units** (4 here). The memory starts as all zeros.

At every time step it does one thing: **combine the current word with the previous memory** to make a new memory.

$$h_t = \tanh(\;x_t \cdot W_x \;+\; h_{t-1} \cdot W_h \;+\; b\;)$$

```
                x_t  (current word: 8 numbers)
                 |
                 v
  h_(t-1) --> [ RNN cell ] --> h_t  --> used at the next step
 (old memory: 4 numbers)     (new memory: 4 numbers)
```

### Following "the movie was not good" through the layer

| t | word | ID | what goes in | combined with | what comes out |
|---|---|---|---|---|---|
| 0 | the | 18 | vector of 8 numbers | starting memory: 4 zeros | new memory $h_0$ (4 numbers) |
| 1 | movie | 14 | vector of 8 numbers | memory from step 0 (4 numbers) | new memory $h_1$ (4 numbers) |
| 2 | was | 20 | vector of 8 numbers | memory from step 1 (4 numbers) | new memory $h_2$ (4 numbers) |
| 3 | not | 15 | vector of 8 numbers | memory from step 2 (4 numbers) | new memory $h_3$ (4 numbers) |
| 4 | good | 9 | vector of 8 numbers | memory from step 3 (4 numbers) | new memory $h_4$ (4 numbers) |

At t = 0 the memory is empty, so only "the" counts. At t = 1 the memory from "the" is mixed with "movie", and so on. By the last step, $h_4$ is a 4-number summary of the whole review, **in order**. That's why "not good" can end up different from "good not".

### A worked illustration with the same sizes as our model

Our layer is `SimpleRNN(4)` receiving word vectors of 8 numbers, so its weights have exactly these shapes:

| Weight | Shape | Count |
|---|---|---|
| $W_x$ | (8, 4): 8 features in, 4 units out | 32 |
| $W_h$ | (4, 4): 4 memory numbers in, 4 out | 16 |
| $b$ | (4,) | 4 |
| **Total** | | **52**, the number `model.summary()` shows |

The values below are **made up for illustration**. The real model's weights start random and change during training, so you won't see these exact numbers in the model. What you *will* see is the same shapes and the same steps.

$W_x$ (one row per feature, one column per unit):

| | unit 1 | unit 2 | unit 3 | unit 4 |
|---|---|---|---|---|
| feature 1 | 0.5 | -0.3 | 0.1 | 0.2 |
| feature 2 | 0.2 | 0.4 | -0.1 | 0.0 |
| feature 3 | 0.1 | 0.1 | 0.3 | -0.2 |
| feature 4 | 0.0 | 0.2 | 0.2 | 0.1 |
| feature 5 | 0.3 | -0.1 | 0.0 | 0.4 |
| feature 6 | -0.2 | 0.3 | 0.1 | 0.1 |
| feature 7 | 0.1 | 0.0 | -0.3 | 0.2 |
| feature 8 | 0.4 | 0.1 | 0.2 | -0.1 |

$W_h$ (one row per old memory number, one column per unit):

| | unit 1 | unit 2 | unit 3 | unit 4 |
|---|---|---|---|---|
| memory 1 | 0.1 | 0.2 | 0.0 | -0.1 |
| memory 2 | -0.1 | 0.3 | 0.1 | 0.0 |
| memory 3 | 0.2 | 0.0 | 0.1 | 0.1 |
| memory 4 | 0.0 | -0.2 | 0.1 | 0.3 |

$b$ = [0.0, 0.1, 0.0, -0.1]

#### Step t = 0, word "the"

The embedding gives $x_0$ = [0.1, 0.2, 0.0, 0.1, 0.3, 0.0, 0.2, 0.1] (8 numbers). The memory starts as $h_{-1}$ = [0, 0, 0, 0].

**1. Current word × $W_x$.** Each unit multiplies all 8 features by its own column and adds them up:

| Unit | Calculation | Result |
|---|---|---|
| 1 | 0.1×0.5 + 0.2×0.2 + 0×0.1 + 0.1×0 + 0.3×0.3 + 0×(-0.2) + 0.2×0.1 + 0.1×0.4 | 0.24 |
| 2 | 0.1×(-0.3) + 0.2×0.4 + 0×0.1 + 0.1×0.2 + 0.3×(-0.1) + 0×0.3 + 0.2×0 + 0.1×0.1 | 0.05 |
| 3 | 0.1×0.1 + 0.2×(-0.1) + 0×0.3 + 0.1×0.2 + 0.3×0 + 0×0.1 + 0.2×(-0.3) + 0.1×0.2 | -0.03 |
| 4 | 0.1×0.2 + 0.2×0 + 0×(-0.2) + 0.1×0.1 + 0.3×0.4 + 0×0.1 + 0.2×0.2 + 0.1×(-0.1) | 0.18 |

So 8 numbers in, 4 numbers out: [0.24, 0.05, -0.03, 0.18].

**2. Old memory × $W_h$.** The memory is all zeros, so this part is [0, 0, 0, 0].

**3. Add the bias:** [0.24, 0.05, -0.03, 0.18] + [0.0, 0.1, 0.0, -0.1] = [0.24, 0.15, -0.03, 0.08]

**4. Squash with tanh:** $h_0$ = **[0.2355, 0.1489, -0.0300, 0.0798]**

#### Step t = 1, word "movie"

Now the memory is no longer zero, so you can see the $W_h$ part working.

$x_1$ = [0.2, 0.0, 0.1, 0.3, 0.1, 0.2, 0.0, 0.1] and $h_0$ = [0.2355, 0.1489, -0.0300, 0.0798].

**1. Current word × $W_x$:**

| Unit | Calculation | Result |
|---|---|---|
| 1 | 0.2×0.5 + 0×0.2 + 0.1×0.1 + 0.3×0 + 0.1×0.3 + 0.2×(-0.2) + 0×0.1 + 0.1×0.4 | 0.14 |
| 2 | 0.2×(-0.3) + 0×0.4 + 0.1×0.1 + 0.3×0.2 + 0.1×(-0.1) + 0.2×0.3 + 0×0 + 0.1×0.1 | 0.07 |
| 3 | 0.2×0.1 + 0×(-0.1) + 0.1×0.3 + 0.3×0.2 + 0.1×0 + 0.2×0.1 + 0×(-0.3) + 0.1×0.2 | 0.15 |
| 4 | 0.2×0.2 + 0×0 + 0.1×(-0.2) + 0.3×0.1 + 0.1×0.4 + 0.2×0.1 + 0×0.2 + 0.1×(-0.1) | 0.10 |

**2. Old memory × $W_h$.** Each unit multiplies the 4 memory numbers by its own column:

| Unit | Calculation | Result |
|---|---|---|
| 1 | 0.2355×0.1 + 0.1489×(-0.1) + (-0.0300)×0.2 + 0.0798×0 | 0.0027 |
| 2 | 0.2355×0.2 + 0.1489×0.3 + (-0.0300)×0 + 0.0798×(-0.2) | 0.0758 |
| 3 | 0.2355×0 + 0.1489×0.1 + (-0.0300)×0.1 + 0.0798×0.1 | 0.0199 |
| 4 | 0.2355×(-0.1) + 0.1489×0 + (-0.0300)×0.1 + 0.0798×0.3 | -0.0026 |

**3. Add everything:** [0.14, 0.07, 0.15, 0.10] + [0.0027, 0.0758, 0.0199, -0.0026] + [0.0, 0.1, 0.0, -0.1] = [0.1427, 0.2458, 0.1699, -0.0026]

**4. Squash with tanh:** $h_1$ = **[0.1417, 0.2410, 0.1683, -0.0026]**

This new memory now carries information from both "the" and "movie". The layer repeats exactly these four steps for "was", "not" and "good", using **the same** $W_x$, $W_h$ and $b$ every time. After t = 4, $h_4$ (4 numbers) is what `SimpleRNN(4)` hands to the Dense layer, which is why `model.summary()` shows its output as `(None, 4)`.

### The whole batch at once

The layer doesn't finish review 1 before starting review 2. At time step $t$ it takes word $t$ from **every review in the batch** together. **Time steps go one after another; reviews in a batch go side by side.**

### The weights, and where the parameter count comes from

| Weight | Shape | Why | Count |
|---|---|---|---|
| $W_x$ (`kernel`) | (8, 4) | features → units | 32 |
| $W_h$ (`recurrent_kernel`) | (4, 4) | old memory → new memory | 16 |
| $b$ (`bias`) | (4,) | one per unit | 4 |
| **Total** | | units × (features + units + 1) = 4 × (8 + 4 + 1) | **52** |

The **same weights are reused at every time step**. A 5-word review and a 500-word review use exactly the same 52 numbers; the layer just loops more times.

### What it hands back: `return_sequences`

The layer computes a memory at every step. What it passes on depends on one setting:

| Setting | Hands back | Shape | Next layer is usually |
|---|---|---|---|
| `return_sequences=False` (default) | only the **last** memory $h_4$ | `(batch, 4)` | Dense |
| `return_sequences=True` | the memory at **every** step | `(batch, 5, 4)` | another RNN/LSTM, or Flatten |

---
# Part 3: Model 1, Embedding → SimpleRNN → Dense

```
input           (None, 5)          word IDs
Embedding       (None, 5, 8)       each ID -> 8 numbers
SimpleRNN(4)    (None, 4)             last memory only (return_sequences=False)
Dense(1)        (None, 1)             one probability per review
```

**Dense(1, sigmoid)** doesn't know anything about words or time steps. It sees 4 numbers per review and turns them into one probability: 4 weights + 1 bias = **5** parameters.

**What to check in `model.summary()` below:**

| Layer | Output Shape | Param # | Why |
|---|---|---|---|
| embedding | (None, 5, 8) | 168 | 21 words × 8 numbers |
| simple_rnn | (None, 4) | 52 | 4 × (8 + 4 + 1) |
| dense | (None, 1) | 5 | 4 weights + 1 bias |

In [ ]:
model_rnn = Sequential([
    Input(shape=(5,)),
    Embedding(input_dim=vocab_size, output_dim=8),
    SimpleRNN(4),
    Dense(1, activation="sigmoid")
])

model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 8)           │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 4)              │            52 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

### Training

With `batch_size=4`, each training step sends **4 reviews** through the model, so the input to the SimpleRNN is `(4, 5, 8)`. 12 reviews / 4 = **3 steps per epoch**. At every step, the embedding table, the RNN weights and the Dense weights all get adjusted a little.

In [ ]:
model_rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model_rnn.fit(X, labels, epochs=30, batch_size=4, verbose=0)

print("Final training accuracy:", history.history["accuracy"][-1])

Final training accuracy: 1.0


### Predictions

`predict` sends all 12 reviews at once, so the batch is 12 and the output shape is `(12, 1)`: one probability per review. Above 0.5 means positive.

With only 12 examples the model is basically memorising, and results can vary a bit. The point here is the shapes and the flow, not the accuracy.

In [ ]:
probs = model_rnn.predict(X, verbose=0)
print("Output shape:", probs.shape)

for review, p, true in zip(reviews, probs, labels):
    print(f"{review:28} P(positive) = {p[0]:.2f}   true label = {true}")

Output shape: (12, 1)
the movie was great          P(positive) = 0.53   true label = 1
the story was very good      P(positive) = 0.73   true label = 1
i loved the acting           P(positive) = 0.56   true label = 1
a great and fun movie        P(positive) = 0.65   true label = 1
the movie was not good       P(positive) = 0.40   true label = 0
the story was boring         P(positive) = 0.38   true label = 0
i hated the acting           P(positive) = 0.37   true label = 0
a boring and slow movie      P(positive) = 0.42   true label = 0
very good fun movie          P(positive) = 0.76   true label = 1
not great at all             P(positive) = 0.27   true label = 0
the acting was very good     P(positive) = 0.78   true label = 1
slow story and bad acting    P(positive) = 0.29   true label = 0


---
# Part 4: Model 2, keeping every time step with `return_sequences=True` + Flatten

Now the RNN hands back its memory at **every** step, not just the last one:

```
input                              (None, 5)
Embedding                          (None, 5, 8)
SimpleRNN(4, return_sequences=True) (None, 5, 4)     5 memories of 4 numbers each
Flatten                            (None, 20)         5 × 4 laid out in one row
Dense(1)                           (None, 1)
```

**Why Flatten?** Dense expects one flat list of numbers per review. `(None, 5, 4)` is a small table per review, so Flatten lays those 5 rows side by side: 5 × 4 = **20** numbers. Flatten has **no parameters**; it only reshapes.

Now the Dense layer sees the memory after every word, not just the final summary, so it needs 20 weights + 1 bias = **21** parameters.

The SimpleRNN still has **52** parameters. `return_sequences` changes **what the layer hands back**, not the weights inside it.

In [ ]:
model_rnn_flat = Sequential([
    Input(shape=(5,)),
    Embedding(input_dim=vocab_size, output_dim=8),
    SimpleRNN(4, return_sequences=True),
    Flatten(),
    Dense(1, activation="sigmoid")
])

model_rnn_flat.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 8)           │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 5, 4)           │            52 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

---
# Part 5: The LSTM layer

## Why a different layer?

The SimpleRNN rewrites its whole memory through tanh at **every** step. In long reviews, early words slowly fade away (this is the **vanishing gradient** problem during training). The LSTM fixes this by keeping **two** memories:

| Memory | Name | Role |
|---|---|---|
| $c$ | **cell state** | long-term memory, a "conveyor belt" that changes only a little at each step |
| $h$ | **hidden state** | short-term memory, and **what the layer hands back** |

Both start as zeros, and both have size = **units** (4 here).

## What happens inside at every time step

**Receives:** the same as the SimpleRNN, `(batch, 5, 8)`, one word vector at a time.

At each step the cell looks at the current word $x_t$ and the previous output $h_{t-1}$, and makes **four** calculations, each with its own weights:

| Part | Symbol | Activation | Values | Question it answers |
|---|---|---|---|---|
| Forget gate | $f$ | sigmoid | 0 to 1 | How much of the old long-term memory should I **keep**? |
| Input gate | $i$ | sigmoid | 0 to 1 | How much new information should I **write**? |
| Candidate | $g$ | tanh | -1 to 1 | **What** is the new information? |
| Output gate | $o$ | sigmoid | 0 to 1 | How much of the memory should I **show** as output? |

A gate at 0 means "block completely"; at 1 means "let everything through".


Only $h$ ($c$ stays inside the layer). The shapes are the same as the SimpleRNN:

`return_sequences=False` → `(batch, 4)`, the last $h$.
`return_sequences=True` → `(batch, 5, 4)`, $h$ at every step.

In [ ]:
model_lstm = Sequential([
    Input(shape=(5,)),
    Embedding(input_dim=vocab_size, output_dim=8),
    LSTM(4),
    Dense(1, activation="sigmoid")
])

model_lstm.summary()

### Training and predictions

Same setup as Model 1, but 60 epochs instead of 30: with 4 times the weights and so little data, the LSTM needs longer to settle. When you look at the predictions, check **"the movie was not good"**. With only 12 examples, the model may still score it above 0.5, because it has seen "good" in positive reviews far more often than "not good". Learning negation is exactly what LSTMs are good at, but it needs far more data than this. On the IMDB notebook's thousands of reviews, this is where the LSTM pulls ahead of the SimpleRNN.

In [ ]:
model_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model_lstm.fit(X, labels, epochs=60, batch_size=4, verbose=0)
print("Final training accuracy:", history.history["accuracy"][-1])

probs = model_lstm.predict(X, verbose=0)
for review, p, true in zip(reviews, probs, labels):
    print(f"{review:28} P(positive) = {p[0]:.2f}   true label = {true}")

---
# Part 6: Model 4, stacking SimpleRNN → LSTM

This is the same pattern as the class IMDB notebook: one recurrent layer feeding another.

```
input                               (None, 5)
Embedding                           (None, 5, 8)
SimpleRNN(4, return_sequences=True)  (None, 5, 4)    a memory for every word
LSTM(4)                             (None, 4)       reads those 5 memories, keeps the last h
Dense(1)                            (None, 1)
```

**Why `return_sequences=True` on the SimpleRNN?** The LSTM needs a **sequence** to read, `(batch, timesteps, features)`. If the SimpleRNN returned only its last memory `(None, 4)`, there would be no time steps left for the LSTM to walk through, and Keras would give an error.

**Notice the LSTM's parameter count changes.** Its input is no longer the 8-number word vectors; it's the SimpleRNN's 4-number memories. So features = 4:
4 × 4 × (4 + 4 + 1) = **144** (instead of 208 in Model 3).

The rule to remember: **the "features" of any layer = the last number in the shape coming out of the layer before it.**

In [ ]:
model_stacked = Sequential([
    Input(shape=(5,)),
    Embedding(input_dim=vocab_size, output_dim=8),
    SimpleRNN(4, return_sequences=True),
    LSTM(4),
    Dense(1, activation="sigmoid")
])

model_stacked.summary()

---
# Part 7: Trying a new review

A new review goes through exactly the same steps: words → IDs → padding → model. It must only use words from our vocabulary, since the model has never seen any others.

The input is **1 review**, so the batch size is 1: the model sees `(1, 5)` and hands back `(1, 1)`.

Your exact probability may differ a little from your neighbour's (different TensorFlow versions train slightly differently), and on a mixed review like this one it may land close to 0.5. The shapes will always be the same.

In [ ]:
new_review = "the acting was not great"

new_ids = [[word_to_id[word] for word in new_review.split()]]
new_X = pad_sequences(new_ids, maxlen=5)
print("Padded input:", new_X, "shape", new_X.shape)

p = model_lstm.predict(new_X, verbose=0)
print("Output shape:", p.shape)
print("P(positive) =", round(float(p[0][0]), 2))

---
# Summary

## The full journey of one batch

```
reviews (text)
    |  words -> IDs, pad to 5
    v
(batch, 5)             word IDs
    |  Embedding: each ID -> 8 learned numbers
    v
(batch, 5, 8)          batch, timesteps, features
    |  SimpleRNN / LSTM with 4 units: read 5 steps, carry a memory of 4 numbers
    v
(batch, 4)                  return_sequences=False: last memory
(batch, 5, 4)            return_sequences=True: memory at every step
    |  (Flatten, if the sequence goes straight to Dense)
    v
(batch, 1)                  Dense + sigmoid: one probability per review
```

## SimpleRNN vs LSTM

| | SimpleRNN | LSTM |
|---|---|---|
| Receives at each step | word $x_t$ + memory $h_{t-1}$ | word $x_t$ + $h_{t-1}$ + long-term memory $c_{t-1}$ |
| Does at each step | one combine + tanh | forget, input, candidate, output; then updates $c$ and $h$ |
| Hands back | $h$ | $h$ ($c$ stays inside) |
| Output shape | `(batch, units)` or `(batch, timesteps, units)` | same |
| Parameters | units × (features + units + 1) | 4 × units × (features + units + 1) |
| Long sequences | early words fade | long-term memory carries them further |

## Practice questions

1. 32 reviews, each padded to 100 words, go into `Embedding(20000, 128)`. What shape comes out?
2. That goes into `SimpleRNN(50, return_sequences=True)`. What shape comes out, and how many parameters?
3. Then `LSTM(100)`. What shape comes out, and how many parameters?
4. In Model 2, what would the Flatten output be if we padded to 10 words instead of 5?
5. Why does `SimpleRNN(4)` have the same number of parameters whether the reviews are 5 words or 500 words long?

*Answers: (1) (32, 100, 128). (2) (32, 100, 50); 50 × (128 + 50 + 1) = 8,950. (3) (32, 100); 4 × 100 × (50 + 100 + 1) = 60,400. (4) 10 × 4 = 40. (5) The same weights are reused at every time step; a longer review only means more loops.*